<a href="https://colab.research.google.com/github/Phils-dev/RAG_Homework/blob/main/PX4_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 드론 운용기록 분석 및 비행안전 기술문서 RAG 지원 시스템

- 과목 : 생성형 AI 활용
- 작성자 :
- 개발환경 : Google Colab, Python
- 대상기체 : Holybro X500 V2 + Pixhawk 6C
- 비행제어 소프트웨어 : PX4 Autopilot
- 기술문서 버전 : PX4 v1.17.0

## 개발 목적

PX4 공식 기술문서를 활용한 RAG 시스템을 구축하고,
드론 운용일지와 점검이력을 분석하여 정비 점검항목 및
비행안전 관련 기술정보를 제공한다.

※ 본 프로그램의 운용일지 및 정비주기 지침은
교육용 가상 데이터이며, 실제 비행 적합성 판단에 사용하지 않는다.

# PART 1. 개발 환경 구성

## 01. 개발 환경 설정

RAG 시스템 구축 및 드론 운용기록 분석에 필요한
Python 패키지를 설치한다.

In [2]:

# 패키지 설치
%pip install -q \
    "langchain>=1.0,<2.0" \
    "langchain-core>=1.4.7,<2.0" \
    "langchain-text-splitters>=1.1.2,<2.0" \
    langchain-openai \
    langchain-chroma \
    chromadb \
    langchain-huggingface \
    sentence-transformers \
    requests==2.32.4 \
    opentelemetry-api==1.42.1 \
    opentelemetry-sdk==1.42.1 \
    opentelemetry-exporter-otlp-proto-grpc==1.42.1 \
    "oauthlib>=3.2.0,<4" \
    jedi


In [2]:
# 패키지 의존성 검사
%pip check

No broken requirements found.


In [3]:
# 설치 환경 확인
import sys
from importlib.metadata import version

# Python 버전 출력
print("Python :", sys.version.split()[0])

# 패키지 목록 정의
packages = [
    "langchain",
    "langchain-core",
    "langchain-text-splitters",
    "langchain-openai",
    "langchain-chroma",
    "chromadb",
    "langchain-huggingface",
    "sentence-transformers",
    "pandas",
    "requests",
    "opentelemetry-api",
    "opentelemetry-sdk"
]

# 패키지 버전 출력
for package in packages:
    print(f"{package} : {version(package)}")

Python : 3.13.18
langchain : 1.4.8
langchain-core : 1.8.5
langchain-text-splitters : 1.2.0
langchain-openai : 1.8.5
langchain-chroma : 1.2.0
chromadb : 1.5.9
langchain-huggingface : 1.3.0
sentence-transformers : 5.7.0
pandas : 2.2.3
requests : 2.32.4
opentelemetry-api : 1.42.1
opentelemetry-sdk : 1.42.1



-----------------------------------------------------------



## 02. 라이브러리 불러오기

RAG 시스템 구축, 기술문서 처리, 운용일지 분석 및
점검 보고서 생성에 필요한 라이브러리를 불러온다.


In [4]:

# 기본 라이브러리
import os
import re
import json
import time
import hashlib
from pathlib import Path

# 데이터 분석 라이브러리
import pandas as pd

# Colab 라이브러리
from google.colab import userdata, files
from IPython.display import display, Markdown

# 문서 처리 라이브러리
from langchain_core.documents import Document
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 임베딩 라이브러리
from langchain_huggingface import HuggingFaceEmbeddings
from transformers import AutoTokenizer

# 벡터 DB 라이브러리
from langchain_chroma import Chroma

# LLM 및 RAG 라이브러리
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 라이브러리 로드 결과 출력
print("라이브러리 로드 완료")


라이브러리 로드 완료



## 03. 실행 환경 설정

OpenAI API 인증키, 임베딩 모델, LLM 모델 및
기술문서와 운용자료의 경로를 설정한다.


In [5]:

# OpenAI API Key 설정
api_key = userdata.get("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OpenAI API Key를 등록하세요.")

os.environ["OPENAI_API_KEY"] = api_key

# API Key 설정 결과 출력
print("OpenAI API Key 설정 완료")


OpenAI API Key 설정 완료


In [6]:

# 기준 기체 설정
AIRCRAFT_MODEL = "Holybro X500 V2"
FLIGHT_CONTROLLER = "Pixhawk 6C"
FLIGHT_SOFTWARE = "PX4 Autopilot"

# PX4 문서 버전 설정
PX4_VERSION = "v1.17.0"

# 임베딩 모델 설정
EMBEDDING_MODEL = "intfloat/multilingual-e5-small"

# LLM 모델 설정
LLM_MODEL = "gpt-5.4-mini"

# 문서 분할 설정
CHUNK_SIZE = 350
CHUNK_OVERLAP = 40
MAX_TOKENS = 512

# 검색 문서 개수 설정
TOP_K = 3


In [7]:

# 프로젝트 경로 설정
PROJECT_DIR = Path("/content/PX4_RAG_Project")

# 세부 디렉터리 경로 설정
DOCUMENT_DIR = PROJECT_DIR / "documents"
DATA_DIR = PROJECT_DIR / "data"
DB_DIR = PROJECT_DIR / "chroma_db"
TEST_DIR = PROJECT_DIR / "tests"
OUTPUT_DIR = PROJECT_DIR / "outputs"

# 디렉터리 목록 정의
directories = [
    DOCUMENT_DIR,
    DATA_DIR,
    DB_DIR,
    TEST_DIR,
    OUTPUT_DIR
]

# 디렉터리 생성
for directory in directories:
    directory.mkdir(parents=True, exist_ok=True)

# 프로젝트 경로 출력
print("프로젝트 디렉터리 생성 완료")
print(PROJECT_DIR)


프로젝트 디렉터리 생성 완료
/content/PX4_RAG_Project


In [8]:

# 실행 환경 설정 결과 출력
print("=" * 60)
print("PX4 RAG 실행 환경")
print("=" * 60)

print("대상 기체       :", AIRCRAFT_MODEL)
print("비행제어기      :", FLIGHT_CONTROLLER)
print("비행제어 SW     :", FLIGHT_SOFTWARE)
print("기술문서 버전   :", PX4_VERSION)
print("임베딩 모델     :", EMBEDDING_MODEL)
print("LLM 모델        :", LLM_MODEL)
print("청크 크기       :", CHUNK_SIZE)
print("청크 중복       :", CHUNK_OVERLAP)
print("최대 토큰       :", MAX_TOKENS)
print("검색 문서 개수  :", TOP_K)

print("-" * 60)

# 디렉터리 생성 상태 확인
for directory in directories:
    status = "정상" if directory.is_dir() else "오류"
    print(f"{directory.name:15s} : {status}")

print("=" * 60)


PX4 RAG 실행 환경
대상 기체       : Holybro X500 V2
비행제어기      : Pixhawk 6C
비행제어 SW     : PX4 Autopilot
기술문서 버전   : v1.17.0
임베딩 모델     : intfloat/multilingual-e5-small
LLM 모델        : gpt-5.4-mini
청크 크기       : 350
청크 중복       : 40
최대 토큰       : 512
검색 문서 개수  : 3
------------------------------------------------------------
documents       : 정상
data            : 정상
chroma_db       : 정상
tests           : 정상
outputs         : 정상



# PART 2. RAG 시스템 구축

## 04. 문서 로드 (Load)

PX4 공식 기술문서를 수집하고 문서의 버전,
출처, 라이선스 및 파일 무결성을 관리한다.

수집된 Markdown 문서를 읽어 RAG 검색에
사용할 Document 객체를 생성한다.

### 4.1. 공식 기술문서 수집


In [9]:

import requests
from datetime import datetime, timezone

# PX4 저장소 설정
PX4_REPOSITORY = "PX4/PX4-Autopilot"
PX4_COMMIT = "d6f12ad1c4f70ad3230afd7d86e971421e02fef4"

# 원본 URL 설정
RAW_URL = (
    f"https://raw.githubusercontent.com/"
    f"{PX4_REPOSITORY}/{PX4_COMMIT}"
)

# 문서 URL 설정
DOCS_URL = "https://docs.px4.io/v1.17/en"

# 라이선스 설정
LICENSE_NAME = "CC BY 4.0"
LICENSE_URL = "https://creativecommons.org/licenses/by/4.0/"

# 수집 대상 문서 정의
DOCUMENT_FILES = [
    "frames_multicopter/holybro_x500v2_pixhawk6c.md",
    "flight_controller/pixhawk6c.md",
    "config/safety_intro.md",
    "config/safety.md",
    "log/flight_review.md",
    "log/flight_log_analysis.md",
    "test_and_ci/test_flights.md"
]

print("수집 대상 문서 :", len(DOCUMENT_FILES))


수집 대상 문서 : 7


In [10]:

# 문서 다운로드 함수 정의
def download_document(url):
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    return response.content

# 문서 수집 결과 정의
downloaded_files = []
failed_files = []

# 공식 문서 다운로드
for relative_path in DOCUMENT_FILES:

    file_name = Path(relative_path).name
    source_path = f"docs/en/{relative_path}"
    source_url = f"{RAW_URL}/{source_path}"

    try:
        content = download_document(source_url)

        # Markdown 내용 확인
        text = content.decode("utf-8")

        if not text.strip():
            raise ValueError("빈 문서")

        # 문서 저장
        file_path = DOCUMENT_DIR / file_name
        file_path.write_bytes(content)

        # 문서 정보 저장
        downloaded_files.append({
            "file_name": file_name,
            "source_path": source_path,
            "source_url": source_url,
            "document_url": (
                f"{DOCS_URL}/{relative_path[:-3]}"
            ),
            "version": PX4_VERSION,
            "commit": PX4_COMMIT,
            "author": "PX4 Documentation Contributors",
            "license": LICENSE_NAME,
            "license_url": LICENSE_URL,
            "source_type": "official",
            "modified": False,
            "sha256": hashlib.sha256(content).hexdigest(),
            "size_bytes": len(content),
            "collected_at": datetime.now(
                timezone.utc
            ).isoformat()
        })

        print(f"[완료] {file_name}")

    except Exception as error:
        failed_files.append(relative_path)
        print(f"[실패] {file_name} : {error}")

# 다운로드 결과 출력
print("-" * 60)
print("다운로드 성공 :", len(downloaded_files))
print("다운로드 실패 :", len(failed_files))


[완료] holybro_x500v2_pixhawk6c.md
[완료] pixhawk6c.md
[완료] safety_intro.md
[완료] safety.md
[완료] flight_review.md
[완료] flight_log_analysis.md
[완료] test_flights.md
------------------------------------------------------------
다운로드 성공 : 7
다운로드 실패 : 0


In [11]:

# 라이선스 원본 URL 설정
license_source_url = f"{RAW_URL}/docs/LICENSE"

# 라이선스 파일 다운로드
license_content = download_document(license_source_url)

# 라이선스 유형 확인
if "CC BY 4.0" not in license_content.decode("utf-8"):
    raise ValueError("라이선스 유형 확인 필요")

# 라이선스 파일 저장
license_path = DOCUMENT_DIR / "LICENSE"
license_path.write_bytes(license_content)

print("라이선스 파일 저장 완료")
print(license_path)


라이선스 파일 저장 완료
/content/PX4_RAG_Project/documents/LICENSE


In [12]:

# 수집 결과 확인
if failed_files:
    raise RuntimeError("다운로드 실패 문서 확인 필요")

# 문서 관리 정보 정의
manifest = {
    "repository": PX4_REPOSITORY,
    "version": PX4_VERSION,
    "commit": PX4_COMMIT,
    "license": LICENSE_NAME,
    "license_source_url": license_source_url,
    "license_sha256": hashlib.sha256(
        license_content
    ).hexdigest(),
    "document_count": len(downloaded_files),
    "documents": downloaded_files
}

# Manifest 파일 경로 설정
manifest_path = DOCUMENT_DIR / "manifest.json"

# Manifest 파일 저장
with open(manifest_path, "w", encoding="utf-8") as file:
    json.dump(
        manifest,
        file,
        ensure_ascii=False,
        indent=4
    )

print("Manifest 파일 저장 완료")


Manifest 파일 저장 완료


In [13]:

# 수집 문서 목록 생성
df_documents = pd.DataFrame(downloaded_files)

# 문서 크기 계산
df_documents["size_kb"] = (
    df_documents["size_bytes"] / 1024
).round(2)

# 문서 목록 출력
display(df_documents[
    ["file_name", "version", "license", "size_kb"]
])

# 문서 수집 결과 출력
print("수집 문서 개수 :", len(df_documents))
print("문서 저장 경로 :", DOCUMENT_DIR)


,file_name,version,license,size_kb
0,holybro_x500v2_pixhawk6c.md,v1.17.0,CC BY 4.0,9.40
1,pixhawk6c.md,v1.17.0,CC BY 4.0,8.37
2,safety_intro.md,v1.17.0,CC BY 4.0,1.22
3,safety.md,v1.17.0,CC BY 4.0,52.67
4,flight_review.md,v1.17.0,CC BY 4.0,21.72
5,flight_log_analysis.md,v1.17.0,CC BY 4.0,8.12
6,test_flights.md,v1.17.0,CC BY 4.0,1.61


수집 문서 개수 : 7
문서 저장 경로 : /content/PX4_RAG_Project/documents


In [14]:

import shutil

# 백업 파일 생성
backup_path = shutil.make_archive(
    "/content/PX4_Documents_v1.17.0",
    "zip",
    root_dir=PROJECT_DIR,
    base_dir="documents"
)

# 백업 파일 다운로드
files.download(backup_path)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


### 4.2. 공식 기술문서 로드

Manifest 정보를 읽고 수집된 문서의 버전 및
파일 무결성을 검증한다.

Markdown 원문과 출처 정보를 LangChain의
Document 객체로 변환한다.


In [15]:

# Manifest 파일 경로 설정
manifest_path = DOCUMENT_DIR / "manifest.json"

# Manifest 파일 로드
with open(manifest_path, "r", encoding="utf-8") as file:
    manifest = json.load(file)

# 문서 버전 확인
if manifest["version"] != PX4_VERSION:
    raise ValueError("PX4 문서 버전 불일치")

# Git commit 확인
if manifest["commit"] != PX4_COMMIT:
    raise ValueError("Git commit 불일치")

# Manifest 정보 출력
print("문서 버전 :", manifest["version"])
print("Git commit :", manifest["commit"])
print("문서 개수 :", manifest["document_count"])
print("라이선스 :", manifest["license"])


문서 버전 : v1.17.0
Git commit : d6f12ad1c4f70ad3230afd7d86e971421e02fef4
문서 개수 : 7
라이선스 : CC BY 4.0


In [16]:

# 문서 무결성 검증 함수 정의
def verify_document(file_path, expected_hash):

    if not file_path.is_file():
        raise FileNotFoundError(file_path.name)

    content = file_path.read_bytes()
    actual_hash = hashlib.sha256(content).hexdigest()

    return actual_hash == expected_hash

# 문서 개수 확인
if len(manifest["documents"]) != manifest["document_count"]:
    raise ValueError("Manifest 문서 개수 불일치")

# 라이선스 파일 검증
if not verify_document(
    DOCUMENT_DIR / "LICENSE",
    manifest["license_sha256"]
):
    raise ValueError("라이선스 파일 변경 확인")

# 문서 무결성 검증
for item in manifest["documents"]:

    file_path = DOCUMENT_DIR / item["file_name"]

    if item["version"] != PX4_VERSION:
        raise ValueError("문서 버전 불일치")

    if item["commit"] != PX4_COMMIT:
        raise ValueError("문서 commit 불일치")

    if not verify_document(file_path, item["sha256"]):
        raise ValueError(f"파일 변경 : {file_path.name}")

    print(f"[정상] {file_path.name}")

print("-" * 60)
print("문서 무결성 검증 완료")


[정상] holybro_x500v2_pixhawk6c.md
[정상] pixhawk6c.md
[정상] safety_intro.md
[정상] safety.md
[정상] flight_review.md
[정상] flight_log_analysis.md
[정상] test_flights.md
------------------------------------------------------------
문서 무결성 검증 완료


In [17]:

# 문서 목록 정의
documents = []

# 공식 기술문서 로드
for item in manifest["documents"]:

    # 문서 경로 설정
    file_path = DOCUMENT_DIR / item["file_name"]

    # Markdown 문서 읽기
    content = file_path.read_text(encoding="utf-8")

    # 문서 제목 추출
    title_match = re.search(
        r"^#\s+(.+)$",
        content,
        re.MULTILINE
    )

    title = (
        title_match.group(1).strip()
        if title_match else file_path.stem
    )

    # 문서 메타데이터 정의
    metadata = {
        "title": title,
        "file_name": item["file_name"],
        "source": item["document_url"],
        "source_url": item["source_url"],
        "version": item["version"],
        "commit": item["commit"],
        "author": item["author"],
        "license": item["license"],
        "license_url": item["license_url"],
        "source_type": item["source_type"],
        "sha256": item["sha256"],
        "modified": item["modified"],
        "collected_at": item["collected_at"]
    }

    # Document 객체 생성
    document = Document(
        page_content=content,
        metadata=metadata
    )

    # 문서 목록 추가
    documents.append(document)

    print(f"[로드 완료] {item['file_name']}")

print("-" * 60)
print("로드된 문서 개수 :", len(documents))


[로드 완료] holybro_x500v2_pixhawk6c.md
[로드 완료] pixhawk6c.md
[로드 완료] safety_intro.md
[로드 완료] safety.md
[로드 완료] flight_review.md
[로드 완료] flight_log_analysis.md
[로드 완료] test_flights.md
------------------------------------------------------------
로드된 문서 개수 : 7


In [18]:

# 문서 정보 목록 정의
document_info = []

# 문서 정보 추출
for index, document in enumerate(documents):

    document_info.append({
        "No": index + 1,
        "문서명": document.metadata["file_name"],
        "제목": document.metadata["title"],
        "문자 수": len(document.page_content),
        "버전": document.metadata["version"],
        "라이선스": document.metadata["license"]
    })

# 문서 목록 생성
df_loaded_documents = pd.DataFrame(document_info)

# 문서 목록 출력
display(df_loaded_documents)

# 문서 로드 결과 출력
print("로드된 문서 수 :", len(documents))
print("전체 문자 수 :", sum(
    len(document.page_content)
    for document in documents
))


,No,문서명,제목,문자 수,버전,라이선스
0,1,holybro_x500v2_pixhawk6c.md,Holybro X500 V2 + Pixhawk 6C (PX4 Dev Kit),9618,v1.17.0,CC BY 4.0
1,2,pixhawk6c.md,Holybro Pixhawk 6C,8558,v1.17.0,CC BY 4.0
2,3,safety_intro.md,Safety,1245,v1.17.0,CC BY 4.0
3,4,safety.md,Safety (Failsafe) Configuration,53925,v1.17.0,CC BY 4.0
4,5,flight_review.md,Log Analysis using Flight Review,22243,v1.17.0,CC BY 4.0
5,6,flight_log_analysis.md,Flight Log Analysis,8315,v1.17.0,CC BY 4.0
6,7,test_flights.md,Tests Flights,1650,v1.17.0,CC BY 4.0


로드된 문서 수 : 7
전체 문자 수 : 105554


In [19]:

# 확인 대상 문서 설정
document_index = 0

# 문서 제목 출력
print("[문서 제목]")
print(documents[document_index].metadata["title"])

# 문서 본문 출력 (전체 문서를 출력하면 내용이 너무 길어지므로 1,000문자만 출력하도록 설정)
print("\n[문서 본문]")
print(documents[document_index].page_content[:1000])

# 문서 메타데이터 출력
print("\n[문서 메타데이터]")
for key, value in documents[document_index].metadata.items():
    print(f"{key} : {value}")


[문서 제목]
Holybro X500 V2 + Pixhawk 6C (PX4 Dev Kit)

[문서 본문]
# Holybro X500 V2 + Pixhawk 6C (PX4 Dev Kit)

This topic provides full instructions for building the [Holybro X500 V2 ARF Kit](https://holybro.com/collections/x500-kits), also known as the Holybro PX4 Dev Kit.

![The fully built vehicle with props removed](../../assets/airframes/multicopter/x500_v2_holybro_pixhawk6c/kit_no_props.jpg)

## Assembly

::: info

- The images in this document can be selected to see a youtube video of the step.
- Each section lists any required screws at the top.
  :::

### Payload and Battery Holder

**Screw**- Sunk Screw M2.5\*6 12pcs

1. Insert the hanger rubber ring gasket in each of their respective hangers.
   Do not use sharp objects to press the rubbers inside.

   [![Assembly1](../../assets/airframes/multicopter/x500_v2_holybro_pixhawk6c/assembly1.png)](https://www.youtube.com/watch?v=4Tid-FCP_aI)

1. Take the battery mounting board and screw it with the slide bar clip using the Sunk Screw M


## 05. 문서 분할 (Split)

PX4 공식 Markdown 문서에서 검색에 불필요한 요소를
정리하고, 제목과 소제목을 기준으로 문서를 분할한다.

분할된 문서는 임베딩 모델의 입력 토큰 수를 고려하여
검색에 적합한 크기로 나누고 출처 정보를 유지한다.


In [20]:

# Markdown 전처리 함수 정의
def preprocess_markdown(content):

    # 줄바꿈 형식 통일
    content = content.replace("\r\n", "\n")
    content = content.replace("\r", "\n")

    # 이미지 링크 제거 및 설명 유지
    content = re.sub(
        r"!\[([^\]]*)\]\([^\n)]*\)",
        r"\1",
        content
    )

    return content.strip()

# 전처리 문서 목록 정의
processed_documents = []

# 문서 전처리 (전처리된 문서는 processed_documents에 별도로 보관)
for document in documents:

    content = preprocess_markdown(
        document.page_content
    )

    metadata = document.metadata.copy()
    metadata["modified"] = True
    metadata["processing"] = "Markdown preprocessing"

    processed_documents.append(
        Document(
            page_content=content,
            metadata=metadata
        )
    )

# 전처리 결과 출력
print("전처리 문서 수 :", len(processed_documents))

print("전처리 이전 문자 수 :", sum(
    len(doc.page_content) for doc in documents
))

print("전처리 이후 문자 수 :", sum(
    len(doc.page_content) for doc in processed_documents
))


전처리 문서 수 : 7
전처리 이전 문자 수 : 105554
전처리 이후 문자 수 : 99431


In [21]:

# 토크나이저 설정
tokenizer = AutoTokenizer.from_pretrained(
    EMBEDDING_MODEL
)

# 토큰 수 계산 함수 정의
def count_tokens(text):

    return len(tokenizer.encode(
        text,
        add_special_tokens=False,
        truncation=False,
        verbose=False
    ))

# 토크나이저 설정 결과 출력
# Hugging Face 관련 warning은 인증 토큰(HF_TOKEN) 없이 접속하여 발생하는 경고 (무시)
print("토크나이저 :", EMBEDDING_MODEL)
print("최대 입력 토큰 :", tokenizer.model_max_length)
print("목표 청크 크기 :", CHUNK_SIZE)
print("청크 중복 크기 :", CHUNK_OVERLAP)


config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

토크나이저 : intfloat/multilingual-e5-small
최대 입력 토큰 : 512
목표 청크 크기 : 350
청크 중복 크기 : 40


In [22]:

# Markdown 제목 분할 기준 정의
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
    ("####", "Header 4"),
    ("#####", "Header 5"),
    ("######", "Header 6")
]

# Markdown 분할기 설정
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    strip_headers=True
)

# 분할 문서 목록 정의
section_documents = []

# Markdown 문서 분할
for document in processed_documents:

    sections = markdown_splitter.split_text(
        document.page_content
    )

    for index, section in enumerate(sections):

        # 기존 메타데이터 복사
        metadata = document.metadata.copy()

        # 제목 메타데이터 추가
        metadata.update(section.metadata)

        # 장·절 경로 생성
        headers = [
            section.metadata.get(f"Header {level}")
            for level in range(1, 7)
        ]

        section_title = " > ".join(
            header for header in headers if header
        )

        # 장·절 정보 설정
        metadata["section"] = (
            section_title or metadata["title"]
        )

        metadata["section_no"] = index + 1

        # 분할 문서 생성
        section_documents.append(
            Document(
                page_content=section.page_content,
                metadata=metadata
            )
        )

# 1차 분할 결과 출력
print("원본 문서 수 :", len(documents))
print("1차 분할 문서 수 :", len(section_documents))


원본 문서 수 : 7
1차 분할 문서 수 : 102


In [23]:

# 문서 분할 정보 목록 정의
section_info = []

# 분할 문서 정보 추출
for index, document in enumerate(section_documents):

    content = document.page_content
    token_count = count_tokens(content)

    # 표 포함 여부 확인
    has_table = bool(re.search(
        r"^\s*\|?\s*:?-{3,}:?\s*\|",
        content,
        re.MULTILINE
    ))

    # 목록 포함 여부 확인
    has_list = bool(re.search(
        r"^\s*(?:[-*+]|\d+\.)\s+",
        content,
        re.MULTILINE
    ))

    # 주의사항 포함 여부 확인
    has_notice = bool(re.search(
        r"^\s*(?:::|>\s*\[!)",
        content,
        re.MULTILINE
    ))

    section_info.append({
        "No": index + 1,
        "문서명": document.metadata["file_name"],
        "장·절": document.metadata["section"],
        "토큰 수": token_count,
        "표 포함": has_table,
        "목록 포함": has_list,
        "주의사항 포함": has_notice,
        "추가 분할": token_count > CHUNK_SIZE
    })

# 분할 정보 데이터프레임 생성
df_sections = pd.DataFrame(section_info)

# 분할 문서 목록 출력
display(df_sections)


,No,문서명,장·절,토큰 수,표 포함,목록 포함,주의사항 포함,추가 분할
0,1,holybro_x500v2_pixhawk6c.md,Holybro X500 V2 + Pixhawk 6C (PX4 Dev Kit),58,False,False,False,False
1,2,holybro_x500v2_pixhawk6c.md,Holybro X500 V2 + Pixhawk 6C (PX4 Dev Kit) > A...,36,False,True,True,False
2,3,holybro_x500v2_pixhawk6c.md,Holybro X500 V2 + Pixhawk 6C (PX4 Dev Kit) > A...,231,False,True,False,False
3,4,holybro_x500v2_pixhawk6c.md,Holybro X500 V2 + Pixhawk 6C (PX4 Dev Kit) > A...,224,False,True,False,False
4,5,holybro_x500v2_pixhawk6c.md,Holybro X500 V2 + Pixhawk 6C (PX4 Dev Kit) > A...,220,False,True,False,False
...,...,...,...,...,...,...,...,...
97,98,flight_log_analysis.md,Flight Log Analysis > Analysis Tools > PX4Tools,225,False,True,False,False
98,99,flight_log_analysis.md,Flight Log Analysis > Analysis Tools > MAVGCL,201,False,True,False,False
99,100,flight_log_analysis.md,Flight Log Analysis > Analysis Tools > Data Co...,93,False,False,False,False
100,101,test_flights.md,Tests Flights,220,False,False,False,False


In [24]:

# 문서별 분할 현황 생성
df_section_summary = df_sections.groupby(
    "문서명"
).agg(
    절_개수=("No", "count"),
    전체_토큰=("토큰 수", "sum"),
    평균_토큰=("토큰 수", "mean"),
    최대_토큰=("토큰 수", "max"),
    추가_분할=("추가 분할", "sum")
).reset_index()

# 평균 토큰 수 정리
df_section_summary["평균_토큰"] = (
    df_section_summary["평균_토큰"].round(1)
)

# 문서별 분할 현황 출력
display(df_section_summary)

# 전체 분할 현황 출력
print("원본 문서 수 :", len(documents))
print("1차 분할 문서 수 :", len(section_documents))
print("전체 토큰 수 :", df_sections["토큰 수"].sum())
print("평균 토큰 수 :", round(
    df_sections["토큰 수"].mean(), 1
))
print("최대 토큰 수 :", df_sections["토큰 수"].max())
print("추가 분할 대상 :", df_sections["추가 분할"].sum())


,문서명,절_개수,전체_토큰,평균_토큰,최대_토큰,추가_분할
0,flight_log_analysis.md,10,2218,221.8,683,1
1,flight_review.md,26,4930,189.6,530,2
2,holybro_x500v2_pixhawk6c.md,15,2279,151.9,500,1
3,pixhawk6c.md,17,2783,163.7,452,3
4,safety.md,31,11735,378.5,1165,15
5,safety_intro.md,1,370,370.0,370,1
6,test_flights.md,2,610,305.0,390,1


원본 문서 수 : 7
1차 분할 문서 수 : 102
전체 토큰 수 : 24925
평균 토큰 수 : 244.4
최대 토큰 수 : 1165
추가 분할 대상 : 24


In [25]:

# 구조적 요소가 포함된 추가 분할 대상 추출
df_review_sections = df_sections[
    df_sections["추가 분할"] &
    (
        df_sections["표 포함"] |
        df_sections["목록 포함"] |
        df_sections["주의사항 포함"]
    )
]

# 추가 검토 대상 출력
display(df_review_sections)

print("구조 보존 검토 대상 :", len(df_review_sections))


,No,문서명,장·절,토큰 수,표 포함,목록 포함,주의사항 포함,추가 분할
12,13,holybro_x500v2_pixhawk6c.md,Holybro X500 V2 + Pixhawk 6C (PX4 Dev Kit) > I...,500,False,True,True,True
16,17,pixhawk6c.md,Holybro Pixhawk 6C > Introduction,362,False,True,False,True
26,27,pixhawk6c.md,Technical Specification > Voltage Ratings,378,False,True,False,True
28,29,pixhawk6c.md,Technical Specification > Debug Port,452,True,True,False,True
32,33,safety_intro.md,Safety,370,False,True,True,True
35,36,safety.md,Safety (Failsafe) Configuration > Failsafe Act...,906,True,False,True,True
38,39,safety.md,Safety (Failsafe) Configuration > Battery Fail...,490,True,False,False,True
39,40,safety.md,Safety (Failsafe) Configuration > Battery Fail...,507,True,True,False,True
40,41,safety.md,Safety (Failsafe) Configuration > Manual Contr...,867,True,False,True,True
41,42,safety.md,Safety (Failsafe) Configuration > Data Link Lo...,353,True,False,False,True


구조 보존 검토 대상 : 24
